# Segmentación de usuarios

**3. Aplicación de algoritmos de clustering**

**3.1 Clustering con K-mean**

In [ ]:
from sklearn.cluster import KMeans

k = 7

kmeans = KMeans(
    n_clusters=k,
    init='k-means++',
    n_init=10,
    max_iter=300,
    random_state=42
)

kmeans.fit(X_scaled)

labels = kmeans.labels_
centroides = kmeans.cluster_centers_
inercia = kmeans.inertia_

print("Inercia final:", inercia)

Se aplicó el algoritmo K means utilizando un número de clusters previamente determinado, con el objetivo de segmentar los datos en grupos homogéneos. Se empleó el método de inicialización k means plus plus, el cual mejora la selección inicial de los centroides y favorece una mejor convergencia del modelo. Asimismo, se configuró un máximo de 300 iteraciones para asegurar que el algoritmo tuviera suficientes oportunidades de estabilizar los centroides. Se utilizaron múltiples inicializaciones mediante el parámetro n init, lo que permite ejecutar el algoritmo varias veces con diferentes puntos de partida y seleccionar la mejor solución en función de la menor inercia. Además, se estableció una semilla aleatoria mediante random state para garantizar la reproducibilidad de los resultados. Estos parámetros en conjunto permiten obtener una segmentación más robusta, evitando soluciones subóptimas y mejorando la calidad del agrupamiento.

La inercia final obtenida fue de 211253.46, lo cual representa la suma de las distancias cuadradas de cada punto a su centroide más cercano, es decir, el grado de compactación de los clusters. Este valor indica qué tan agrupados se encuentran los datos dentro de cada cluster; a menor inercia, mayor cohesión interna. En este caso, la inercia refleja un nivel adecuado de agrupamiento considerando la complejidad y dimensionalidad del conjunto de datos, aunque su interpretación debe complementarse con otras métricas como el silhouette score y el índice de Davies Bouldin para evaluar de manera más integral la calidad del modelo.

In [ ]:
centroides_df = pd.DataFrame(centroides, columns=X.columns)

In [ ]:
from sklearn.preprocessing import StandardScaler

centroides_originales = scaler.inverse_transform(centroides)

centroides_originales_df = pd.DataFrame(centroides_originales, columns=X.columns)

In [ ]:
centroides_df = pd.DataFrame(
    kmeans.cluster_centers_,
    columns=X.columns
)

centroides_df.index = [f'Cluster {i}' for i in range(len(centroides_df))]

centroides_df

El análisis de los centroides obtenidos mediante el algoritmo K means permitió identificar distintos perfiles de comportamiento entre los usuarios. Se observa que los clusters 5 y 1 representan a los usuarios de mayor valor, destacando especialmente el cluster 5 por sus altos niveles de interacción, clics, compras y conversión, lo que indica un alto grado de compromiso con la plataforma. Por otro lado, los clusters 2 y 6 agrupan usuarios con alta actividad reflejada en un elevado número de clics, productos vistos y días activos pero con baja conversión, lo que sugiere oportunidades de mejora en estrategias de persuasión o experiencia de usuario. En contraste, los clusters 0, 3 y 4 corresponden a usuarios con baja interacción, escaso nivel de compras y menor participación general, considerados como segmentos pasivos o de bajo impacto. Los resultados evidencian una segmentación clara que permite identificar tanto a los usuarios más rentables como a aquellos que requieren estrategias específicas para incrementar su nivel de engagement y conversión.

In [ ]:
from sklearn.metrics import silhouette_score
from scipy.spatial.distance import pdist
from sklearn.metrics import davies_bouldin_score


sil_score = silhouette_score(X_scaled, labels)
print("Silhouette Score:", sil_score)

print("Intra-cluster distance (Inercia):", inercia)

inter_cluster_dist = pdist(centroides)
print("Distancias entre centroides:", inter_cluster_dist)

db_index = davies_bouldin_score(X_scaled, labels)
print("Davies-Bouldin Index:", db_index)

Estas métricas confirman que la segmentación obtenida es de buena calidad, logrando un equilibrio adecuado entre cohesión interna y separación externa de los clusters.

In [ ]:
from sklearn.decomposition import PCA

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
centroides_pca = pca.transform(centroides)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,6))

# puntos
plt.scatter(X_pca[:,0], X_pca[:,1], c=labels, s=5)

# centroides
plt.scatter(centroides_pca[:,0], centroides_pca[:,1], marker='X', s=200)

plt.title("Clusters con K-means")


plt.show()

Los clusters presentan en general una buena separación en el espacio reducido mediante PCA, lo que indica que el modelo K means logró identificar grupos con características diferenciadas. Se observan clusters claramente definidos, particularmente en las regiones derecha inferior del gráfico, donde la distancia entre grupos es mayor. No obstante, existe cierto grado de solapamiento en la zona izquierda, lo que sugiere que algunos usuarios comparten características similares y podrían no estar completamente diferenciados. En conjunto, la segmentación es adecuada, aunque con áreas donde la separación entre clusters podría mejorarse.

In [ ]:
# Conteo
conteo = pd.Series(labels).value_counts().sort_index()

# Porcentaje
porcentaje = (conteo / len(labels)) * 100

# Tabla final
tabla_clusters = pd.DataFrame({
    'Número de usuarios': conteo,
    'Porcentaje (%)': porcentaje
})

tabla_clusters

El análisis del tamaño de los clusters muestra una distribución claramente desbalanceada entre los grupos identificados. El cluster 4 concentra la mayor proporción de usuarios con un 46.60%, seguido del cluster 3 con un 24.39% y el cluster 2 con un 13.01%, lo que indica que la mayoría de los usuarios se agrupan en estos tres segmentos principales.

**3.2 Clustering con Mean-Shift**

In [ ]:
from sklearn.cluster import estimate_bandwidth

# muestra 10,000 puntos
sample = X_scaled[np.random.choice(len(X_scaled), size=10000, replace=False)]

bandwidth_02 = estimate_bandwidth(sample, quantile=0.2)
bandwidth_03 = estimate_bandwidth(sample, quantile=0.3)
bandwidth_04 = estimate_bandwidth(sample, quantile=0.4)


print("Bandwidth q=0.2:", bandwidth_02)
print("Bandwidth q=0.3:", bandwidth_03)
print("Bandwidth q=0.4:", bandwidth_04)

Debido al tamaño del dataset (200,000 observaciones), se utilizó una muestra aleatoria de 10,000 datos para estimar el parámetro bandwidth, reduciendo el costo computacional sin afectar significativamente la representatividad de los resultados.

El parámetro bandwidth fue estimado utilizando la función estimate bandwidth con distintos valores de quantile (0.2, 0.3 y 0.4). Se seleccionó el valor correspondiente a quantile = 0.3, obteniendo un bandwidth de aproximadamente 2.94, ya que proporciona un equilibrio adecuado entre la cantidad de clusters generados y la separación entre los mismos, evitando tanto la sobresegmentación como la generalización excesiva de los datos.

In [ ]:
from sklearn.cluster import MeanShift
sample = X_scaled[np.random.choice(len(X_scaled), size=10000, replace=False)]

ms = MeanShift(bandwidth=2.94)
ms.fit(sample)

labels_sample = ms.labels_
centros = ms.cluster_centers_

n_clusters = len(set(labels_sample))

print("Clusters:", n_clusters)

Según la muestra, el algoritmo Mean Shift identificó un total de 36 clusters, lo cual es significativamente mayor en comparación con los 7 clusters obtenidos mediante K means. Esto indica que Mean Shift es más sensible a la densidad de los datos y tiende a generar una segmentación más detallada, identificando subgrupos dentro de los clusters principales.

In [ ]:
from sklearn.metrics import silhouette_score, davies_bouldin_score
from scipy.spatial.distance import pdist

# Silhouette
sil_ms = silhouette_score(sample, labels_sample)

# Intra-cluster (como K-means)
inercia_ms = sum(
    ((sample[labels_sample == i] - centros[i])**2).sum()
    for i in range(n_clusters)
)

# Inter-cluster
inter_ms = pdist(centros)

# Davies-Bouldin
db_ms = davies_bouldin_score(sample, labels_sample)

print("Silhouette:", sil_ms)
print("Intra-cluster:", inercia_ms)
print("Inter-cluster:", inter_ms)
print("Davies-Bouldin:", db_ms)

Se obtuvo un silhouette score de 0.46, lo que indica una separación moderada entre los clusters, inferior a la obtenida con K means, sugiriendo una menor claridad en la delimitación de los grupos. La distancia intra cluster fue de 44502.21, reflejando una buena compactación interna considerando el alto número de clusters generados. Por otro lado, las distancias inter cluster presentan una amplia variabilidad, con valores que van desde distancias pequeñas hasta separaciones considerablemente grandes, lo que indica que algunos clusters están bien diferenciados mientras que otros se encuentran más cercanos entre sí. El índice de Davies Bouldin de 0.71, al ser menor a 1, indica un buen desempeño del modelo en términos de cohesión y separación.

In [ ]:
import matplotlib.pyplot as plt

# Proyectar los datos de la muestra a las 2 componentes principales
sample_pca = pca.transform(sample)

# proyectar centros
centros_pca = pca.transform(centros)

plt.figure(figsize=(8,6))

# puntos de la muestra
plt.scatter(sample_pca[:,0], sample_pca[:,1], c=labels_sample, s=5)

# centros
plt.scatter(
    centros_pca[:,0],
    centros_pca[:,1],
    marker='X',
    s=100,
    c=range(n_clusters)
)

plt.title("Clusters con Mean-Shift")


plt.show()

In [ ]:

# Conteo
conteo_ms = pd.Series(labels_sample).value_counts().sort_index()

# Porcentaje
porcentaje_ms = (conteo_ms / len(labels_sample)) * 100

# Tabla final
tabla_ms = pd.DataFrame({
    'Número de usuarios': conteo_ms,
    'Porcentaje (%)': porcentaje_ms
})

tabla_ms

Mean Shift encontró más clusters que el valor de k seleccionado en K means, ya que identificó 36 clusters frente a los 7 definidos previamente. Esto evidencia que Mean Shift es más sensible a la densidad de los datos y tiende a generar una segmentación más detallada, detectando subgrupos dentro de los clusters principales.

La distribución de usuarios por cluster obtenida mediante Mean Shift evidencia una alta concentración en pocos grupos principales, destacando el cluster 0 que agrupa el 74% del total.

En cuanto a la comparación observada por las gráficas, los clusters obtenidos son más fragmentados y menos homogéneos que los de K means, presentando formas irregulares y una mayor dispersión. Aunque algunas regiones del espacio coinciden con las agrupaciones generales identificadas por K means, en general los resultados son más granulares y diferentes, lo que podría ofrecer mayor detalle pero dificulta su interpretación y aplicación práctica.

3.3 Clustering con DBSCAN

In [ ]:
from sklearn.neighbors import NearestNeighbors

k = 4

neighbors = NearestNeighbors(n_neighbors=k)
neighbors_fit = neighbors.fit(X_scaled)
distances, indices = neighbors_fit.kneighbors(X_scaled)

k_distances = np.sort(distances[:, k-1])

plt.figure(figsize=(8,5))
plt.plot(k_distances)
plt.title("K-distance Graph (k=4)")
plt.xlabel("Puntos ordenados")
plt.ylabel("Distancia")
plt.show()

En este caso, el codo se ubicó aproximadamente en un valor de 0.4, por lo que se seleccionó eps = 0.4 como el parámetro óptimo. Este valor permite un adecuado equilibrio entre la formación de clusters y la detección de ruido, evitando tanto la sobreagrupación como la fragmentación excesiva de los datos.

In [ ]:
plt.figure(figsize=(8,5))
plt.plot(k_distances)

eps = 0.4
plt.axhline(y=eps, color='red', linestyle='--', label=f'eps = {eps}')

plt.title("K-distance Graph (k=4)")
plt.xlabel("Puntos ordenados")
plt.ylabel("Distancia")
plt.legend()

plt.show()

In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score
import numpy as np

eps = 0.4

resultados = []

for min_s in [3, 4, 5]:
    db = DBSCAN(eps=eps, min_samples=min_s)
    labels_db = db.fit_predict(X_scaled)

    # número de clusters (sin ruido)
    n_clusters = len(set(labels_db)) - (1 if -1 in labels_db else 0)

    # ruido
    ruido = list(labels_db).count(-1)
    porcentaje_ruido = (ruido / len(labels_db)) * 100

    # quitar ruido para métricas
    mask = labels_db != -1
    if len(set(labels_db[mask])) > 1:
        sil = silhouette_score(X_scaled[mask], labels_db[mask])
        db_index = davies_bouldin_score(X_scaled[mask], labels_db[mask])
    else:
        sil = None
        db_index = None

    resultados.append({
        "min_samples": min_s,
        "clusters": n_clusters,
        "ruido (%)": porcentaje_ruido,
        "silhouette": sil,
        "davies_bouldin": db_index
    })

import pandas as pd
tabla_dbscan = pd.DataFrame(resultados)
tabla_dbscan

Se observó que al incrementar el parámetro min samples, el número de clusters disminuye, mientras que el porcentaje de ruido aumenta ligeramente. El mejor desempeño se obtuvo con min samples = 5, ya que presentó el mayor silhouette score, lo que indica una mejor separación entre los clusters. Aunque el índice de Davies Bouldin fue ligeramente superior en comparación con otros valores, se mantiene dentro de un rango aceptable. Por lo tanto, este valor ofrece un equilibrio adecuado entre la calidad del agrupamiento, la reducción de clusters excesivos y un nivel controlado de puntos clasificados como ruido.

In [ ]:
from sklearn.metrics import silhouette_score, davies_bouldin_score
from scipy.spatial.distance import pdist

mask = labels_db != -1
X_filtrado = X_scaled[mask]
labels_filtrado = labels_db[mask]

centroides_db = np.array([
    X_filtrado[labels_filtrado == i].mean(axis=0)
    for i in set(labels_filtrado)
])



# intra
intra_db = sum(
    ((X_filtrado[labels_filtrado == i] - centroides_db[j])**2).sum()
    for j, i in enumerate(set(labels_filtrado))
)

# inter
inter_db = pdist(centroides_db)

print("Intra:", intra_db)
print("Inter:", inter_db)

In [ ]:
plt.figure(figsize=(8,6))

# puntos normales
plt.scatter(X_pca[labels_db != -1, 0], X_pca[labels_db != -1, 1], c=labels_db[labels_db != -1])

# ruido
plt.scatter(X_pca[labels_db == -1, 0], X_pca[labels_db == -1, 1],
            c='pink', marker='x', label='Ruido')

plt.title("Clusters con DBSCAN")
plt.legend()

plt.show()

La visualización de los resultados obtenidos con DBSCAN muestra la formación de clusters de tamaño reducido junto con una cantidad considerable de puntos clasificados como ruido. Esto indica que el algoritmo es más estricto en la definición de grupos, identificando únicamente regiones de alta densidad y descartando aquellos puntos que no cumplen con los criterios establecidos. En comparación con K means y Mean Shift, DBSCAN permite detectar estructuras más complejas y excluir outliers, lo que proporciona una segmentación más robusta en presencia de datos atípicos.

In [ ]:
# Conteo
conteo_db = pd.Series(labels_db).value_counts().sort_index()

# Porcentaje
porcentaje_db = (conteo_db / len(labels_db)) * 100

# Tabla
tabla_db = pd.DataFrame({
    'Número de usuarios': conteo_db,
    'Porcentaje (%)': porcentaje_db
})

tabla_db

Los usuarios clasificados como ruido por DBSCAN corresponden principalmente a datos atípicos, ya que presentan comportamientos distintos al resto y no se agrupan en regiones de alta densidad. Estos usuarios se caracterizan por patrones irregulares o extremos en sus interacciones, lo que los separa de los clusters principales. Aunque es más probable que representen anomalías dentro del dataset, no se descarta que algunos puedan corresponder a bots, usuarios nuevos o comportamientos poco comunes.